# Pipeline Validation & Automated Checks

This notebook provides implementation evidence for PP1.

It does **not** perform the final research evaluation.

The purpose is to verify that the current C3 development pipeline:

- produces structurally valid outputs;
- keeps values inside expected ranges;
- creates one temporal row per Product × Aspect × Quarter;
- produces one latest row per Product × Aspect;
- generates valid trend / emerging-status / risk labels;
- can recover at least some intentionally planted synthetic temporal changes.

All findings here relate only to the **synthetic dummy development data**.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

WORKSPACE = Path.cwd()
if WORKSPACE.name == "notebooks":
    COMPONENT_ROOT = WORKSPACE.parent
elif (WORKSPACE / "data" / "processed").exists():
    COMPONENT_ROOT = WORKSPACE
else:
    COMPONENT_ROOT = WORKSPACE / "component_3"

PROCESSED = COMPONENT_ROOT / "data" / "processed"
PIPELINE_INPUT = PROCESSED / "paq_with_emerging_status.csv"
LATEST_ASPECT_INPUT = PROCESSED / "pp1_c3_latest_aspect_output.csv"
PRODUCT_SUMMARY_INPUT = PROCESSED / "pp1_c3_product_summary.csv"
GROUND_TRUTH = PROCESSED / "dummy_c1_ground_truth.csv"

TABLE_DIR = COMPONENT_ROOT / "outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load development outputs


In [2]:
pipeline = pd.read_csv(PIPELINE_INPUT)
latest = pd.read_csv(LATEST_ASPECT_INPUT)
products = pd.read_csv(PRODUCT_SUMMARY_INPUT)

print("Pipeline rows:", len(pipeline))
print("Latest aspect rows:", len(latest))
print("Product summary rows:", len(products))


Pipeline rows: 550
Latest aspect rows: 50
Product summary rows: 10


## 2. Structural checks

These checks verify that the pipeline produced the expected data structure.


In [3]:
results = []

def record(test_name, passed, detail):
    results.append({
        "test_name": test_name,
        "passed": bool(passed),
        "detail": detail
    })

# Required final columns
required_latest = [
    "parent_asin",
    "aspect",
    "latest_quarter",
    "review_perceived_quality_dev",
    "trend",
    "change_point",
    "emerging_status",
    "emerging_label",
    "issue_risk",
    "confidence_dev"
]

missing = [c for c in required_latest if c not in latest.columns]

record(
    "Latest-output schema",
    len(missing) == 0,
    "Missing columns: " + str(missing)
)

# One row per PAQ
duplicates = pipeline.duplicated(
    subset=["parent_asin", "aspect", "quarter"]
).sum()

record(
    "Unique Product-Aspect-Quarter rows",
    duplicates == 0,
    f"Duplicate PAQ rows: {duplicates}"
)

# One latest row per product-aspect
latest_duplicates = latest.duplicated(
    subset=["parent_asin", "aspect"]
).sum()

record(
    "One latest row per Product-Aspect",
    latest_duplicates == 0,
    f"Duplicate latest rows: {latest_duplicates}"
)

# Product count consistency
product_count_match = (
    latest["parent_asin"].nunique()
    == products["parent_asin"].nunique()
)

record(
    "Product count consistency",
    product_count_match,
    f"Latest products={latest['parent_asin'].nunique()}, "
    f"summary products={products['parent_asin'].nunique()}"
)

pd.DataFrame(results)


,test_name,passed,detail
0,Latest-output schema,True,Missing columns: []
1,Unique Product-Aspect-Quarter rows,True,Duplicate PAQ rows: 0
2,One latest row per Product-Aspect,True,Duplicate latest rows: 0
3,Product count consistency,True,"Latest products=10, summary products=10"


## 3. Range and label checks


In [4]:
record(
    "Quality is within 0-100",
    latest["review_perceived_quality_dev"].between(0, 100).all(),
    f"Min={latest['review_perceived_quality_dev'].min()}, "
    f"Max={latest['review_perceived_quality_dev'].max()}"
)

record(
    "Confidence is within 0-1",
    latest["confidence_dev"].between(0, 1).all(),
    f"Min={latest['confidence_dev'].min()}, "
    f"Max={latest['confidence_dev'].max()}"
)

valid_trends = {"Improving", "Stable", "Declining"}

record(
    "Trend labels are valid",
    set(latest["trend"].dropna()).issubset(valid_trends),
    "Observed: " + str(sorted(latest["trend"].dropna().unique()))
)

valid_status = {0, 1, 2}

record(
    "Emerging-status values are valid",
    set(latest["emerging_status"].dropna().astype(int)).issubset(valid_status),
    "Observed: " + str(sorted(latest["emerging_status"].dropna().unique()))
)

valid_risks = {"Low", "Medium", "High"}

record(
    "Risk labels are valid",
    set(latest["issue_risk"].dropna()).issubset(valid_risks),
    "Observed: " + str(sorted(latest["issue_risk"].dropna().unique()))
)


## 4. Logical consistency checks

These are simple rule-level checks for the current development implementation.


In [5]:
# Status 2 should never have Low risk in the current rule set.
bad_status2_risk = latest[
    (latest["emerging_status"] == 2)
    & (latest["issue_risk"] == "Low")
]

record(
    "Emerging status 2 is not Low risk",
    len(bad_status2_risk) == 0,
    f"Violations: {len(bad_status2_risk)}"
)

# Status 0 should normally have Low risk under current rules.
bad_status0_risk = latest[
    (latest["emerging_status"] == 0)
    & (latest["issue_risk"] != "Low")
]

record(
    "Status 0 maps to Low risk",
    len(bad_status0_risk) == 0,
    f"Violations: {len(bad_status0_risk)}"
)

# Confidence intervals should be ordered correctly.
if {
    "quality_ci_lower_dev",
    "review_perceived_quality_dev",
    "quality_ci_upper_dev"
}.issubset(latest.columns):
    ci_ok = (
        (latest["quality_ci_lower_dev"] <= latest["quality_ci_upper_dev"])
        & latest["quality_ci_lower_dev"].between(0, 100)
        & latest["quality_ci_upper_dev"].between(0, 100)
    ).all()

    record(
        "Quality confidence intervals are valid",
        ci_ok,
        "Lower <= Upper and both within 0-100"
    )


## 5. Synthetic planted-change check

The dummy dataset contains known synthetic temporal changes.

This test does **not** measure final research accuracy.

It simply checks whether the development pipeline reacts to known injected changes.


In [6]:
if GROUND_TRUTH.exists():

    truth = pd.read_csv(GROUND_TRUTH)

    latest_eval = latest.merge(
        truth[[
            "parent_asin",
            "aspect",
            "pattern",
            "change_quarter",
            "complaint_theme"
        ]],
        on=["parent_asin", "aspect"],
        how="inner"
    )

    def quarter_distance(q1, q2):
        if pd.isna(q1) or pd.isna(q2):
            return np.nan

        try:
            p1 = pd.Period(str(q1), freq="Q")
            p2 = pd.Period(str(q2), freq="Q")
            return abs(p1.ordinal - p2.ordinal)
        except Exception:
            return np.nan

    latest_eval["change_distance_q"] = latest_eval.apply(
        lambda row: quarter_distance(
            row["change_point"],
            row["change_quarter"]
        ),
        axis=1
    )

    latest_eval["change_detected_near_truth"] = (
        latest_eval["change_distance_q"] <= 1
    )

    latest_eval["decline_pattern_expected"] = (
        latest_eval["pattern"]
        .isin(["decline", "mild_decline", "emerging_issue"])
    )

    latest_eval["direction_reaction"] = np.where(
        latest_eval["decline_pattern_expected"],
        latest_eval["trend"].eq("Declining")
        | latest_eval["emerging_status"].ge(1),
        True
    )

    latest_eval.to_csv(
        TABLE_DIR / "pp1_synthetic_case_validation.csv",
        index=False
    )

    display(
        latest_eval[[
            "parent_asin",
            "aspect",
            "pattern",
            "change_quarter",
            "change_point",
            "change_distance_q",
            "trend",
            "emerging_status",
            "issue_risk",
            "direction_reaction"
        ]]
    )

    print(
        "Planted cases checked:",
        len(latest_eval)
    )

    print(
        "Cases with nearby detected change point:",
        int(latest_eval["change_detected_near_truth"].sum())
    )

    print(
        "Decline/emerging cases with a negative reaction signal:",
        int(
            latest_eval.loc[
                latest_eval["decline_pattern_expected"],
                "direction_reaction"
            ].sum()
        ),
        "/",
        int(latest_eval["decline_pattern_expected"].sum())
    )

else:
    print(
        "Ground-truth file not found. "
        "Place dummy_c1_ground_truth.csv in data/processed/ "
        "and rerun this section."
    )


Ground-truth file not found. Place dummy_c1_ground_truth.csv in data/processed/ and rerun this section.


## 6. Final automated-check report


In [7]:
test_report = pd.DataFrame(results)

test_report.to_csv(
    TABLE_DIR / "pp1_pipeline_test_report.csv",
    index=False
)

display(test_report)

passed = int(test_report["passed"].sum())
total = len(test_report)

print(f"Passed: {passed}/{total}")

if passed == total:
    print("All automated structural and logical checks passed.")
else:
    print("One or more checks need inspection.")


,test_name,passed,detail
0,Latest-output schema,True,Missing columns: []
1,Unique Product-Aspect-Quarter rows,True,Duplicate PAQ rows: 0
2,One latest row per Product-Aspect,True,Duplicate latest rows: 0
3,Product count consistency,True,"Latest products=10, summary products=10"
4,Quality is within 0-100,True,"Min=35.5, Max=82.8"
5,Confidence is within 0-1,True,"Min=0.464, Max=0.787"
6,Trend labels are valid,True,"Observed: ['Declining', 'Stable']"
7,Emerging-status values are valid,True,"Observed: [np.int64(0), np.int64(1), np.int64(2)]"
8,Risk labels are valid,True,"Observed: ['High', 'Low', 'Medium']"
9,Emerging status 2 is not Low risk,True,Violations: 0


Passed: 12/12
All automated structural and logical checks passed.


# PP1 interpretation

If these checks pass, you can say:

> I verified the implementation using automated structural and logical tests, and I also used synthetic cases with intentionally planted temporal changes to confirm that the pipeline reacts to known changes before integrating the real Component 1 output.

Do **not** say:

> The model achieved final accuracy / F1.

because this notebook does not provide final research evaluation.

## Evidence generated

- `outputs/tables/pp1_pipeline_test_report.csv`
- `outputs/tables/pp1_synthetic_case_validation.csv` (if ground truth is available)

## Suggested Git commit

`Add PP1 pipeline validation tests`

## Next step

After this validation notebook:
- capture GitHub / test / output evidence;
- prepare the PP1 demo flow;
- prepare architecture and progress slides;
- replace dummy input with real C1 output when available.
